# Phase 1: Data Preprocessing for Olympic Games


## 1. Import Libraries


In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

csv_path = '/Users/raj/Downloads/athlete_events.csv'

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load the Dataset


In [32]:
print("Loading data...")
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print("First 5 rows of the raw data (One row = One Athlete in one Event):")
df.head()

Loading data...
Dataset shape: (271116, 15)
First 5 rows of the raw data (One row = One Athlete in one Event):


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


## 3. Data Cleaning — Handle Missing Values


In [33]:
print("Missing values in Medal column before cleaning:", df['Medal'].isnull().sum())

df['Medal'] = df['Medal'].fillna('None')

print("Missing values after cleaning:", df['Medal'].isnull().sum())
df[['Name', 'Event', 'Medal']].head(10)

Missing values in Medal column before cleaning: 231333
Missing values after cleaning: 0


,Name,Event,Medal
0,A Dijiang,Basketball Men's Basketball,None
1,A Lamusi,Judo Men's Extra-Lightweight,None
2,Gunnar Nielsen Aaby,Football Men's Football,None
3,Edgar Lindenau Aabye,Tug-Of-War Men's Tug-Of-War,Gold
4,Christine Jacoba Aaftink,Speed Skating Women's 500 metres,None
5,Christine Jacoba Aaftink,"Speed Skating Women's 1,000 metres",None
6,Christine Jacoba Aaftink,Speed Skating Women's 500 metres,None
7,Christine Jacoba Aaftink,"Speed Skating Women's 1,000 metres",None
8,Christine Jacoba Aaftink,Speed Skating Women's 500 metres,None
9,Christine Jacoba Aaftink,"Speed Skating Women's 1,000 metres",None


## 4. Data Transformation — One-Hot Encoding


In [34]:
medal_dummies = pd.get_dummies(df['Medal'], prefix='Medal', dtype=int)

if 'Medal_None' in medal_dummies.columns:
    medal_dummies = medal_dummies.drop('Medal_None', axis=1)

df = pd.concat([df, medal_dummies], axis=1)

print("Notice the 3 new Medal binary columns added:")
df[['Name', 'Medal', 'Medal_Bronze', 'Medal_Silver', 'Medal_Gold']].head(10)

Notice the 3 new Medal binary columns added:


,Name,Medal,Medal_Bronze,Medal_Silver,Medal_Gold
0,A Dijiang,None,0,0,0
1,A Lamusi,None,0,0,0
2,Gunnar Nielsen Aaby,None,0,0,0
3,Edgar Lindenau Aabye,Gold,0,0,1
4,Christine Jacoba Aaftink,None,0,0,0
5,Christine Jacoba Aaftink,None,0,0,0
6,Christine Jacoba Aaftink,None,0,0,0
7,Christine Jacoba Aaftink,None,0,0,0
8,Christine Jacoba Aaftink,None,0,0,0
9,Christine Jacoba Aaftink,None,0,0,0


## 5. Data Aggregation — Country-Year Level


In [35]:

country_year_data = df.groupby(['NOC', 'Year']).agg(
    Athlete_Participation_Count=('ID', 'nunique'),
    Event_Count=('Event', 'nunique'),
    Total_Gold=('Medal_Gold', 'sum'),
    Total_Silver=('Medal_Silver', 'sum'),
    Total_Bronze=('Medal_Bronze', 'sum')
).reset_index()

country_year_data['Total_Medals'] = (
    country_year_data['Total_Gold'] + 
    country_year_data['Total_Silver'] + 
    country_year_data['Total_Bronze']
)

print(f"New Aggregated Dataset Shape: {country_year_data.shape}")
print("Every row represents ONE country's overall performance in ONE specific year!")
country_year_data.head(10)

New Aggregated Dataset Shape: (3305, 8)
Every row represents ONE country's overall performance in ONE specific year!


,NOC,Year,Athlete_Participation_Count,Event_Count,Total_Gold,Total_Silver,Total_Bronze,Total_Medals
0,AFG,1936,15,4,0,0,0,0
1,AFG,1948,25,2,0,0,0,0
2,AFG,1956,12,1,0,0,0,0
3,AFG,1960,12,13,0,0,0,0
4,AFG,1964,8,8,0,0,0,0
5,AFG,1968,5,5,0,0,0,0
6,AFG,1972,8,8,0,0,0,0
7,AFG,1980,11,11,0,0,0,0
8,AFG,1988,5,5,0,0,0,0
9,AFG,1996,2,2,0,0,0,0


## 6. Data Normalization — StandardScaler


In [36]:

scaler = StandardScaler()

numeric_columns = ['Athlete_Participation_Count', 'Event_Count']
country_year_data_scaled = country_year_data.copy()
country_year_data_scaled[numeric_columns] = scaler.fit_transform(country_year_data_scaled[numeric_columns])

print("Final Normalized Data for Machine Learning (Phase 1 Complete!):")
country_year_data_scaled.head(10)

Final Normalized Data for Machine Learning (Phase 1 Complete!):


,NOC,Year,Athlete_Participation_Count,Event_Count,Total_Gold,Total_Silver,Total_Bronze,Total_Medals
0,AFG,1936,-0.438660,-0.641638,0,0,0,0
1,AFG,1948,-0.333478,-0.683554,0,0,0,0
2,AFG,1956,-0.470215,-0.704511,0,0,0,0
3,AFG,1960,-0.470215,-0.453020,0,0,0,0
4,AFG,1964,-0.512287,-0.557808,0,0,0,0
5,AFG,1968,-0.543842,-0.620681,0,0,0,0
6,AFG,1972,-0.512287,-0.557808,0,0,0,0
7,AFG,1980,-0.480733,-0.494935,0,0,0,0
8,AFG,1988,-0.543842,-0.620681,0,0,0,0
9,AFG,1996,-0.575396,-0.683554,0,0,0,0


## 7. Exploratory Data Analysis (EDA)


In [ ]:
print(country_year_data.describe())
print()
print('Medal distribution across all country-years:')
print(country_year_data[['Total_Gold','Total_Silver','Total_Bronze','Total_Medals']].sum())
print()
top10 = country_year_data.groupby('NOC')['Total_Medals'].sum().sort_values(ascending=False).head(10)
print('Top 10 countries by total medals:')
print(top10)
print()
yearly_trend = country_year_data.groupby('Year')[['Total_Medals','Athlete_Participation_Count']].sum()
print('Yearly participation and medal trends:')
print(yearly_trend.tail(10))


## 8. Data Visualization — Bar Charts, Line Plots & Histograms


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

top10.plot(kind='bar', ax=axes[0,0], color='goldenrod')
axes[0,0].set_title('Top 10 Countries by Total Medals')
axes[0,0].set_ylabel('Medals')

yearly_trend['Total_Medals'].plot(ax=axes[0,1])
axes[0,1].set_title('Total Medals Awarded Over Years')
axes[0,1].set_ylabel('Medals')

yearly_trend['Athlete_Participation_Count'].plot(ax=axes[1,0], color='green')
axes[1,0].set_title('Athlete Participation Over Years')
axes[1,0].set_ylabel('Athletes')

country_year_data['Total_Medals'].hist(bins=30, ax=axes[1,1], color='coral')
axes[1,1].set_title('Distribution of Medal Counts')
axes[1,1].set_xlabel('Medals')

plt.tight_layout()
plt.show()


## 9. Correlation Analysis & Heatmap


In [ ]:
numeric_cols = country_year_data[['Athlete_Participation_Count','Event_Count','Total_Gold','Total_Silver','Total_Bronze','Total_Medals']]
corr_matrix = numeric_cols.corr()
print(corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()


## 10. Feature Selection — PCA (Principal Component Analysis)


In [ ]:
from sklearn.decomposition import PCA

X_pca_input = country_year_data_scaled[['Athlete_Participation_Count','Event_Count']]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_pca_input)
print('PCA explained variance ratio:', pca.explained_variance_ratio_)
print('Total variance explained:', sum(pca.explained_variance_ratio_))

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=country_year_data['Total_Medals'], cmap='viridis', alpha=0.5)
plt.colorbar(label='Total Medals')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of Country-Year Features')
plt.tight_layout()
plt.show()


## 11. Feature Selection — Forward & Backward Selection


In [38]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

# We use Linear Regression as the base for our feature selection
lr = LinearRegression()

# Forward Selection (starts empty and adds best features)
X_full = country_year_data[['Athlete_Participation_Count', 'Event_Count', 'Total_Gold', 'Total_Silver', 'Total_Bronze']]
sfs_forward = SequentialFeatureSelector(lr, n_features_to_select=2, direction='forward')
sfs_forward.fit(X_full, country_year_data['Total_Medals'])
print("Top features chosen by Forward Selection:", X_full.columns[sfs_forward.get_support()].tolist())

# Backward Selection (starts with all and removes the worst features)
sfs_backward = SequentialFeatureSelector(lr, n_features_to_select=2, direction='backward')
sfs_backward.fit(X_full, country_year_data['Total_Medals'])
print("Top features chosen by Backward Selection:", X_full.columns[sfs_backward.get_support()].tolist())


Top features chosen by Forward Selection: ['Total_Gold', 'Total_Bronze']
Top features chosen by Backward Selection: ['Total_Gold', 'Total_Bronze']


## 12. Model Training, Classification & F1 Score Evaluation


In [39]:
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Train the Regression Model (using Athlete and Event count to avoid cheating with medal columns)
selected_features = ['Athlete_Participation_Count', 'Event_Count']
X_train, X_test, y_train, y_test = train_test_split(country_year_data_scaled[selected_features], 
                                                    country_year_data['Total_Medals'], 
                                                    test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred_continuous = reg_model.predict(X_test)

# 2. Classify Performance into High / Medium / Low
def categorize_performance(medals):
    if medals >= 10:
        return 'High'
    elif medals >= 3:
        return 'Medium'
    else:
        return 'Low'

# Apply the classification logic to both the real answers and the AI's predictions
# We use np.maximum to ensure no negative medal predictions mess up the logic
y_test_categories = [categorize_performance(m) for m in y_test]
y_pred_categories = [categorize_performance(m) for m in np.maximum(y_pred_continuous, 0)]

# 3. Evaluate the effectiveness using F1 Score
f1 = f1_score(y_test_categories, y_pred_categories, average='weighted')

print(f"Final Model F1 Score: {f1:.4f}\n")
print("Detailed Classification Report:")
print(classification_report(y_test_categories, y_pred_categories))


Final Model F1 Score: 0.7823

Detailed Classification Report:
              precision    recall  f1-score   support

        High       0.76      0.85      0.80       149
         Low       0.89      0.85      0.87       431
      Medium       0.27      0.28      0.28        81

    accuracy                           0.78       661
   macro avg       0.64      0.66      0.65       661
weighted avg       0.79      0.78      0.78       661

